# Smartphone Addiction: Baseline + Phase 3 Tuning

Playground Series S6E8 — Phases 2–3 of `docs/2_implementation_plan.md`.

Phase 2: sanity baselines, native-categorical strong models, `_is_missing`
ablation, engineered features, class-weight A/B.

Phase 3 (this extension): small hand-designed LightGBM / CatBoost / HGB
parameter search, engineered-feature A/B on the best tuned family, and a
paired-bootstrap promotion gate derived from this dataset's own OOF fold
variance (`docs/4_codex_claude_review_log.md` §13.4). Optuna, XGBoost, and
ensembling stay config-gated and off by default.


## 1. Config

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
N_FOLDS = 5
N_BOOTSTRAP = 2000
np.random.seed(SEED)

# Mode flags: one per experiment block, per docs/0_coding_standards.md.
# Phase 2 blocks default off so a Phase 3 re-run stays focused; flip True
# to reproduce the Phase 2 ledger end-to-end.
RUN_V1_SANITY = False
RUN_V2_STRONG = False
RUN_V2_MISSING_ABLATION = False
RUN_V3_ENGINEERED = False
RUN_CLASS_WEIGHT_ABLATION = False

# Phase 3 — evidence-gated sequence (docs/2_implementation_plan.md).
RUN_V4_HAND_TUNED = True
RUN_V4_ENGINEERED_AB = True
RUN_V4_PROMOTION_GATE = True
RUN_V4_OPTUNA = False          # enable only if hand-search shows headroom
RUN_V4_XGBOOST = False         # enable only if complementary residuals
RUN_V4_ENSEMBLE = False        # enable only if diversity + paired evidence

# Smoke-data guard: refuse promotion claims on synthetic local data.
DATA_IS_SMOKE = Path("../data/SMOKE_DATA_ONLY.txt").exists() or Path("data/SMOKE_DATA_ONLY.txt").exists()

pd.set_option("display.max_columns", 50)
print(f"DATA_IS_SMOKE={DATA_IS_SMOKE}")

## 2. Data Loading

In [ ]:
if os.path.exists("/kaggle/input/playground-series-s6e8"):
    DATA_DIR = "/kaggle/input/playground-series-s6e8"
else:
    DATA_DIR = "../data"

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

TARGET = "addicted_label"
NUMERIC_FEATURES = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
CATEGORICAL_FEATURES = ["gender", "stress_level", "academic_work_impact"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = train[ALL_FEATURES].copy()
y = train[TARGET].copy()
X_test = test[ALL_FEATURES].copy()

for col in CATEGORICAL_FEATURES:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

print(f"X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")

## 3. Cross-Validation Helper

In [ ]:
results = []  # (name, oof_auc, fold_aucs) for the summary table
oof_store = {}  # name -> oof predictions, for sanity checks / future ensembling

def run_cv(name: str, fit_predict_fold, X_df: pd.DataFrame, y_ser: pd.Series) -> np.ndarray:
    """Run stratified 5-fold CV, print per-fold and overall OOF AUC.

    Args:
        name: label for the results table.
        fit_predict_fold: callable(X_tr, y_tr, X_val) -> val_pred_proba.
        X_df: feature frame.
        y_ser: target series.

    Returns:
        OOF prediction array aligned to X_df's row order.
    """
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X_df))
    fold_aucs = []
    start = time.time()
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_df, y_ser)):
        X_tr, X_val = X_df.iloc[tr_idx], X_df.iloc[val_idx]
        y_tr, y_val = y_ser.iloc[tr_idx], y_ser.iloc[val_idx]
        val_pred = fit_predict_fold(X_tr, y_tr, X_val)
        oof[val_idx] = val_pred
        fold_auc = roc_auc_score(y_val, val_pred)
        fold_aucs.append(fold_auc)
    overall_auc = roc_auc_score(y_ser, oof)
    elapsed = time.time() - start
    print(
        f"{name:40s} OOF AUC={overall_auc:.5f}  "
        f"fold std={np.std(fold_aucs):.5f}  ({elapsed:.0f}s)"
    )
    results.append({
        "name": name,
        "oof_auc": overall_auc,
        "fold_auc_mean": np.mean(fold_aucs),
        "fold_auc_std": np.std(fold_aucs),
        "fold_aucs": fold_aucs,
    })
    oof_store[name] = oof
    return oof

## 4. v1 — Sanity Baselines

Constant predictor, logistic regression, and `HistGradientBoostingClassifier` establish the floor and confirm the evaluation pipeline before any tuning, per `docs/2_implementation_plan.md` Phase 2 step 2.

In [ ]:
if RUN_V1_SANITY:
    # Constant predictor: no ranking signal by construction, AUC = 0.5
    # (not computed via roc_auc_score, which requires score variation);
    # recorded directly as the theoretical floor.
    results.append({
        "name": "v1a_constant", "oof_auc": 0.5,
        "fold_auc_mean": 0.5, "fold_auc_std": 0.0, "fold_aucs": [0.5] * N_FOLDS,
    })
    print(f"{'v1a_constant':40s} OOF AUC=0.50000  (theoretical floor, not fit)")

In [ ]:
if RUN_V1_SANITY:
    def fit_predict_logreg(X_tr, y_tr, X_val):
        pipe = Pipeline([
            ("prep", ColumnTransformer([
                ("num", Pipeline([
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]), NUMERIC_FEATURES),
                ("cat", Pipeline([
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]), CATEGORICAL_FEATURES),
            ])),
            ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
        ])
        pipe.fit(X_tr, y_tr)
        return pipe.predict_proba(X_val)[:, 1]

    _ = run_cv("v1b_logistic_regression", fit_predict_logreg, X, y)

In [ ]:
if RUN_V1_SANITY:
    def fit_predict_hgb(X_tr, y_tr, X_val):
        model = HistGradientBoostingClassifier(
            random_state=SEED, max_iter=200, categorical_features="from_dtype"
        )
        model.fit(X_tr, y_tr)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v1c_hist_gradient_boosting", fit_predict_hgb, X, y)

**Insight:** the constant predictor's AUC=0.5 confirms the floor; logistic regression and HGB's actual OOF AUCs (see the summary table in Section 8) confirm the pipeline is producing genuine ranking signal well above that floor before any tuning — exact numbers in `docs/6_baseline_modeling.md`.

## 5. v2 — Strong Models (Native Categorical)

In [ ]:
if RUN_V2_STRONG:
    def fit_predict_lgbm(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    lgbm_oof = run_cv("v2a_lightgbm_native_cat", fit_predict_lgbm, X, y)

In [ ]:
if RUN_V2_STRONG:
    def catboost_ready(df: pd.DataFrame) -> pd.DataFrame:
        # CatBoost's pandas-Categorical cat_features path rejects NaN
        # directly ("cat_features must be integer or string ... NaN
        # values should be converted to string") -- give it an explicit
        # "missing" string category instead, still distinct from every
        # real level, so this is native-missing-handling in spirit even
        # though LightGBM/HGB can take the NaN itself.
        out = df.copy()
        for col in CATEGORICAL_FEATURES:
            out[col] = out[col].astype("object").fillna("missing").astype(str)
        return out

    def fit_predict_catboost(X_tr, y_tr, X_val):
        model = CatBoostClassifier(
            random_seed=SEED, iterations=200, depth=6, learning_rate=0.05,
            cat_features=CATEGORICAL_FEATURES, verbose=False,
        )
        model.fit(catboost_ready(X_tr), y_tr)
        return model.predict_proba(catboost_ready(X_val))[:, 1]

    catboost_oof = run_cv("v2b_catboost_native_cat", fit_predict_catboost, X, y)

**Insight:** compare against v1's sanity baselines in `docs/6_baseline_modeling.md` — native categorical + native missing-value handling should clear the HGB floor if the tree ensembles are extracting more signal than a single boosting pass on ordinal-ish encodings.

## 6. `_is_missing` Indicator Flags — OOF Ablation

Per `docs/3_eda_insights.md` §4.2/§10: the marginal analysis in EDA found no strong target signal in missingness, but explicitly did not rule out a conditional effect. This is the actual test — LightGBM with vs. without explicit `_is_missing` columns alongside native NaN handling, same model/fold setup as v2a for a clean comparison.

In [ ]:
if RUN_V2_MISSING_ABLATION:
    X_with_flags = X.copy()
    for col in ALL_FEATURES:
        X_with_flags[f"{col}_is_missing"] = X[col].isna().astype(int)

    def fit_predict_lgbm_flags(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v2c_lightgbm_plus_missing_flags", fit_predict_lgbm_flags, X_with_flags, y)

**Insight:** compare `v2a_lightgbm_native_cat` vs. `v2c_lightgbm_plus_missing_flags` OOF AUC in `docs/6_baseline_modeling.md` — this is the direct answer to whether `_is_missing` flags earn their place, not the marginal EDA table.

## 7. v3 — Engineered Features

EDA-informed ratio/residual features among the three strongest predictors (`docs/3_eda_insights.md` §3/§10), computed identically on train and test, target-free (`docs/0_coding_standards.md` leakage rule):

In [ ]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add EDA-informed ratio/residual features. Target-free, safe to
    compute once outside the CV loop (docs/0_coding_standards.md)."""
    out = df.copy()
    out["social_to_screen_ratio"] = df["social_media_hours"] / df["daily_screen_time_hours"].replace(0, np.nan)
    out["gaming_to_screen_ratio"] = df["gaming_hours"] / df["daily_screen_time_hours"].replace(0, np.nan)
    out["time_budget_residual"] = 24 - (
        df["sleep_hours"] + df["work_study_hours"] + df["daily_screen_time_hours"]
    )
    out["weekend_escalation"] = df["weekend_screen_time"] - df["daily_screen_time_hours"]
    return out

ENGINEERED_FEATURES = [
    "social_to_screen_ratio", "gaming_to_screen_ratio",
    "time_budget_residual", "weekend_escalation",
]

if RUN_V3_ENGINEERED:
    X_engineered = add_engineered_features(X)

    def fit_predict_lgbm_engineered(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v3_lightgbm_plus_engineered", fit_predict_lgbm_engineered, X_engineered, y)

**Insight:** compare `v3_lightgbm_plus_engineered` against `v2a_lightgbm_native_cat` — ratios/residuals of features a tree ensemble can already split on nonlinearly are not guaranteed to help; exact delta in `docs/6_baseline_modeling.md`.

## 8. Class-Imbalance A/B

AUC is rank-based, so `class_weight` mainly affects optimizer dynamics rather than the final ranking — tested directly rather than assumed either way, per `docs/2_implementation_plan.md` Phase 2 step 5.

In [ ]:
if RUN_CLASS_WEIGHT_ABLATION:
    def fit_predict_lgbm_balanced(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1, class_weight="balanced",
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v2d_lightgbm_class_weight_balanced", fit_predict_lgbm_balanced, X, y)

**Insight:** compare against `v2a_lightgbm_native_cat` (unweighted) — expect little to no AUC change either way since AUC only depends on score ranking; exact numbers in `docs/6_baseline_modeling.md`.

## 9. Summary and Candidate Sanity Checks

In [ ]:
summary = pd.DataFrame(results)[["name", "oof_auc", "fold_auc_mean", "fold_auc_std"]]
summary = summary.sort_values("oof_auc", ascending=False).reset_index(drop=True)
summary

In [ ]:
def candidate_sanity_checks(name: str, oof_pred: np.ndarray, y_true: pd.Series) -> dict:
    """Replaces the old predicted-rate-vs-70.94% check (which assumed a
    threshold AUC optimization doesn't make) per
    docs/2_implementation_plan.md Phase 2 step 6."""
    finite_in_range = bool(np.all(np.isfinite(oof_pred)) and np.all((oof_pred >= 0) & (oof_pred <= 1)))
    n_unique = int(pd.Series(oof_pred).nunique())
    overall_auc = roc_auc_score(y_true, oof_pred)
    return {
        "name": name,
        "finite_in_[0,1]": finite_in_range,
        "n_unique_predictions": n_unique,
        "prediction_range": (float(oof_pred.min()), float(oof_pred.max())),
        "overall_oof_auc": overall_auc,
    }

best_name = summary.iloc[0]["name"] if summary.iloc[0]["name"] != "v1a_constant" else summary.iloc[1]["name"]
checks = candidate_sanity_checks(best_name, oof_store[best_name], y)
pd.Series(checks)

**Insight:** the leading candidate (excluding the constant floor) passes basic sanity (finite, bounded, non-degenerate predictions) before being considered for Phase 3 promotion-gate comparison. Full progression table and interpretation in `docs/6_baseline_modeling.md`.

## 10. Phase 3 — Hand-Designed Parameter Search


Per `docs/2_implementation_plan.md` Phase 3 step 2 and
`docs/6_baseline_modeling.md` §8: Phase 2's HGB > LightGBM/CatBoost gap was a
**floor-setting artifact of mismatched hyperparameters**, not a model-family
verdict. This section runs a small hand-designed grid that aligns learning
rates and iteration budgets across HGB / LightGBM / CatBoost before any
family comparison.

Configs are deliberately few (not Optuna). `_is_missing` flags and
`class_weight` are **not** tuning dimensions (Phase 2 resolved both).


In [ ]:
def make_lgbm_fit(params: dict):
    """Return a fold fit_predict callable for LightGBM with given params."""

    def fit_predict(X_tr, y_tr, X_val):
        model = LGBMClassifier(random_state=SEED, verbose=-1, **params)
        model.fit(
            X_tr, y_tr, categorical_feature=[c for c in CATEGORICAL_FEATURES if c in X_tr.columns]
        )
        return model.predict_proba(X_val)[:, 1]

    return fit_predict


def make_hgb_fit(params: dict):
    def fit_predict(X_tr, y_tr, X_val):
        model = HistGradientBoostingClassifier(
            random_state=SEED, categorical_features="from_dtype", **params
        )
        model.fit(X_tr, y_tr)
        return model.predict_proba(X_val)[:, 1]

    return fit_predict


def make_cb_fit(params: dict):
    def fit_predict(X_tr, y_tr, X_val):
        model = CatBoostClassifier(
            random_seed=SEED,
            cat_features=[c for c in CATEGORICAL_FEATURES if c in X_tr.columns],
            verbose=False,
            **params,
        )
        model.fit(catboost_ready(X_tr), y_tr)
        return model.predict_proba(catboost_ready(X_val))[:, 1]

    return fit_predict


# Phase 2 reference points (re-run for a fair same-fold comparison store).
V4_CONFIGS = [
    # --- HGB ---
    ("v4a_hgb_phase2_ref", "hgb", {"max_iter": 200}),  # sklearn default lr=0.1
    ("v4b_hgb_lr05_iter400", "hgb", {"learning_rate": 0.05, "max_iter": 400}),
    ("v4c_hgb_lr05_iter600_leaf31", "hgb", {
        "learning_rate": 0.05, "max_iter": 600, "max_leaf_nodes": 31
    }),
    # --- LightGBM ---
    ("v4d_lgbm_match_hgb", "lgbm", {
        "n_estimators": 200, "num_leaves": 31, "learning_rate": 0.1
    }),
    ("v4e_lgbm_phase2_ref", "lgbm", {
        "n_estimators": 200, "num_leaves": 31, "learning_rate": 0.05
    }),
    ("v4f_lgbm_deeper", "lgbm", {
        "n_estimators": 500, "num_leaves": 63, "learning_rate": 0.05,
        "min_child_samples": 20,
    }),
    ("v4g_lgbm_regularized", "lgbm", {
        "n_estimators": 800, "num_leaves": 31, "learning_rate": 0.03,
        "min_child_samples": 50, "subsample": 0.8, "colsample_bytree": 0.8,
    }),
    ("v4h_lgbm_shallow", "lgbm", {
        "n_estimators": 600, "num_leaves": 15, "learning_rate": 0.05,
        "min_child_samples": 40,
    }),
    # --- CatBoost ---
    ("v4i_cb_match_hgb", "cb", {"iterations": 200, "depth": 6, "learning_rate": 0.1}),
    ("v4j_cb_phase2_ref", "cb", {"iterations": 200, "depth": 6, "learning_rate": 0.05}),
    ("v4k_cb_more_iters", "cb", {"iterations": 600, "depth": 6, "learning_rate": 0.05}),
    ("v4l_cb_deeper", "cb", {"iterations": 400, "depth": 8, "learning_rate": 0.05}),
    ("v4m_cb_shallow_long", "cb", {"iterations": 800, "depth": 4, "learning_rate": 0.03}),
]

if RUN_V4_HAND_TUNED:
    # catboost_ready may be missing if RUN_V2_STRONG was False — define here.
    if "catboost_ready" not in globals():
        def catboost_ready(df: pd.DataFrame) -> pd.DataFrame:
            out = df.copy()
            for col in CATEGORICAL_FEATURES:
                if col in out.columns:
                    out[col] = out[col].astype("object").fillna("missing").astype(str)
            return out

    makers = {"hgb": make_hgb_fit, "lgbm": make_lgbm_fit, "cb": make_cb_fit}
    for name, family, params in V4_CONFIGS:
        run_cv(name, makers[family](params), X, y)

**Insight:** compare matched-budget configs (`v4a`/`v4d`/`v4i` at lr≈0.1,
200 trees) before reading family rankings. If the best configs within a family
cluster within fold-std of each other, the hand search has plateaued and Optuna
is not justified (`RUN_V4_OPTUNA` stays False).


## 11. Phase 3 — Engineered Features A/B On Best Tuned Model


Phase 2's engineered pack was directionally positive (+0.0005) but
smaller than fold std. Re-test on the best hand-tuned LightGBM (native cats +
engineered) versus the same params on raw features — promotion still requires
the paired bootstrap in §12.


In [ ]:
# Ensure engineered helper exists even when RUN_V3_ENGINEERED was False.
if "add_engineered_features" not in globals():
    def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        daily = out["daily_screen_time_hours"].replace(0, np.nan)
        out["social_to_screen_ratio"] = out["social_media_hours"] / daily
        out["gaming_to_screen_ratio"] = out["gaming_hours"] / daily
        out["time_budget_residual"] = 24 - (
            out["sleep_hours"] + out["work_study_hours"] + out["daily_screen_time_hours"]
        )
        out["weekend_escalation"] = out["weekend_screen_time"] - out["daily_screen_time_hours"]
        return out


if RUN_V4_ENGINEERED_AB and RUN_V4_HAND_TUNED:
    v4_rows = [r for r in results if r["name"].startswith("v4") and "engineered" not in r["name"]]
    if not v4_rows:
        print("No v4 results to A/B against.")
    else:
        best_v4 = max(v4_rows, key=lambda r: r["oof_auc"])
        best_name = best_v4["name"]
        best_cfg = next(c for c in V4_CONFIGS if c[0] == best_name)
        _, family, params = best_cfg
        print(f"Best hand-tuned so far: {best_name} OOF={best_v4['oof_auc']:.5f}")

        makers_local = {"hgb": make_hgb_fit, "lgbm": make_lgbm_fit, "cb": make_cb_fit}
        X_eng = add_engineered_features(X)
        eng_name = f"{best_name}_plus_engineered"
        run_cv(eng_name, makers_local[family](params), X_eng, y)

        raw_auc = best_v4["oof_auc"]
        eng_auc = next(r["oof_auc"] for r in results if r["name"] == eng_name)
        print(f"Engineered delta vs raw-tuned: {eng_auc - raw_auc:+.6f}")

## 12. Phase 3 — Paired-Bootstrap Promotion Gate


Promotion gate from `docs/2_implementation_plan.md` Phase 3 step 7
(replacing the borrowed `+0.0002` threshold):

1. Measure fold-to-fold OOF AUC std among candidates.
2. Paired bootstrap on OOF predictions (same rows) vs. current champion.
3. Require fold consistency (gain not driven by a single fold).

Smoke-data runs report numbers for pipeline verification only — they must not
be treated as competition promotion decisions (`DATA_IS_SMOKE`).


In [ ]:
def paired_bootstrap_auc_delta(
    y_true: np.ndarray,
    pred_a: np.ndarray,
    pred_b: np.ndarray,
    n_boot: int = N_BOOTSTRAP,
    seed: int = SEED,
) -> dict:
    """Bootstrap the AUC(pred_b) - AUC(pred_a) paired difference."""
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    n = len(y_true)
    deltas = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        # Skip degenerate resamples with a single class.
        if y_true[idx].min() == y_true[idx].max():
            deltas[i] = np.nan
            continue
        deltas[i] = roc_auc_score(y_true[idx], pred_b[idx]) - roc_auc_score(
            y_true[idx], pred_a[idx]
        )
    deltas = deltas[~np.isnan(deltas)]
    return {
        "mean_delta": float(np.mean(deltas)),
        "ci_low": float(np.quantile(deltas, 0.025)),
        "ci_high": float(np.quantile(deltas, 0.975)),
        "p_positive": float(np.mean(deltas > 0)),
        "n_boot_used": int(len(deltas)),
    }


def fold_auc_deltas(
    y_true: pd.Series, pred_a: np.ndarray, pred_b: np.ndarray
) -> list[float]:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    deltas = []
    for _, val_idx in skf.split(np.zeros(len(y_true)), y_true):
        deltas.append(
            roc_auc_score(y_true.iloc[val_idx], pred_b[val_idx])
            - roc_auc_score(y_true.iloc[val_idx], pred_a[val_idx])
        )
    return deltas


if RUN_V4_PROMOTION_GATE and oof_store:
    summary_v4 = (
        pd.DataFrame(results)
        .sort_values("oof_auc", ascending=False)
        .reset_index(drop=True)
    )
    display_cols = ["name", "oof_auc", "fold_auc_mean", "fold_auc_std"]
    print("=== All results (sorted) ===")
    print(summary_v4[display_cols].to_string(index=False))

    # Champion = best non-constant OOF currently in store.
    ranked = [r for r in results if r["name"] != "v1a_constant"]
    champion = max(ranked, key=lambda r: r["oof_auc"])
    # Reference challenger pool: everything within 0.002 of champion for gate detail.
    challengers = [
        r for r in ranked
        if r["name"] != champion["name"] and champion["oof_auc"] - r["oof_auc"] <= 0.002
    ]
    # Always compare top-2 even if gap > 0.002.
    second = sorted(ranked, key=lambda r: r["oof_auc"], reverse=True)[1]
    if second["name"] not in {c["name"] for c in challengers}:
        challengers.append(second)

    print(f"\nChampion: {champion['name']} OOF={champion['oof_auc']:.5f} "
          f"fold_std={champion['fold_auc_std']:.5f}")
    if DATA_IS_SMOKE:
        print("WARNING: DATA_IS_SMOKE=True — numbers are pipeline verification only.")

    gate_rows = []
    champ_oof = oof_store[champion["name"]]
    for chall in challengers:
        boot = paired_bootstrap_auc_delta(y.to_numpy(), champ_oof, oof_store[chall["name"]])
        # Direction: challenger - champion already in boot when pred_b=chall.
        # Recompute as chall - champ explicitly:
        boot = paired_bootstrap_auc_delta(
            y.to_numpy(), champ_oof, oof_store[chall["name"]]
        )
        fold_d = fold_auc_deltas(y, champ_oof, oof_store[chall["name"]])
        # For rows where chall is worse, mean_delta will be negative — fine.
        # Also evaluate each as candidate vs champion when chall is better:
        row = {
            "candidate": chall["name"],
            "candidate_oof": chall["oof_auc"],
            "delta_vs_champion": chall["oof_auc"] - champion["oof_auc"],
            "boot_mean_delta": boot["mean_delta"],
            "boot_ci_low": boot["ci_low"],
            "boot_ci_high": boot["ci_high"],
            "boot_p_positive": boot["p_positive"],
            "fold_deltas": [round(d, 6) for d in fold_d],
            "n_folds_positive": int(sum(d > 0 for d in fold_d)),
        }
        gate_rows.append(row)
        print(
            f"  vs {chall['name']}: ΔOOF={row['delta_vs_champion']:+.5f} "
            f"bootΔ={boot['mean_delta']:+.5f} "
            f"CI=[{boot['ci_low']:+.5f},{boot['ci_high']:+.5f}] "
            f"P(Δ>0)={boot['p_positive']:.3f} "
            f"folds+={row['n_folds_positive']}/{N_FOLDS}"
        )

    gate_df = pd.DataFrame(gate_rows)
    promotion_decision = {
        "champion": champion["name"],
        "champion_oof": champion["oof_auc"],
        "data_is_smoke": DATA_IS_SMOKE,
        "optuna_justified": False,
        "xgboost_justified": False,
        "ensemble_justified": False,
    }

    # Optuna headroom: within-family spread of hand-designed configs.
    family_spreads = {}
    for family_prefix, label in [("v4a_hgb", "hgb"), ("v4b_hgb", "hgb"), ("v4c_hgb", "hgb"),
                                  ("v4d_lgbm", "lgbm"), ("v4e_lgbm", "lgbm"), ("v4f_lgbm", "lgbm"),
                                  ("v4g_lgbm", "lgbm"), ("v4h_lgbm", "lgbm"),
                                  ("v4i_cb", "cb"), ("v4j_cb", "cb"), ("v4k_cb", "cb"),
                                  ("v4l_cb", "cb"), ("v4m_cb", "cb")]:
        pass
    def _family_of(name: str):
        if "hgb" in name:
            return "hgb"
        if "lgbm" in name:
            return "lgbm"
        if "_cb_" in name:
            return "cb"
        return None

    by_family_rows = {"hgb": [], "lgbm": [], "cb": []}
    for r in results:
        fam = _family_of(r["name"])
        if fam and r["name"].startswith("v4") and "engineered" not in r["name"]:
            by_family_rows[fam].append(r)
    for fam, rows in by_family_rows.items():
        if len(rows) >= 2:
            spread = max(r["oof_auc"] for r in rows) - min(r["oof_auc"] for r in rows)
            family_spreads[fam] = spread
            ordered = sorted((r["oof_auc"] for r in rows), reverse=True)
            top_gap = ordered[0] - ordered[1]
            print(f"Family {fam}: spread={spread:.5f} top_gap={top_gap:.5f}")

    family_top_gaps = {}
    for fam, rows in by_family_rows.items():
        if len(rows) >= 2:
            ordered = sorted((r["oof_auc"] for r in rows), reverse=True)
            family_top_gaps[fam] = ordered[0] - ordered[1]

    # Optuna only if top-of-grid is still contested (best vs second-best), not
    # merely because a weak failed config widened min-max spread.
    fold_noise = champion["fold_auc_std"]
    promotion_decision["family_spreads"] = family_spreads
    promotion_decision["family_top_gaps"] = family_top_gaps
    promotion_decision["fold_noise"] = fold_noise
    if family_top_gaps and max(family_top_gaps.values()) > max(0.0010, 1.5 * fold_noise):
        promotion_decision["optuna_justified"] = True
        print("Optuna gate: OPEN (top-of-grid still contested beyond fold noise).")
    else:
        print("Optuna gate: CLOSED (hand-search top plateaus within fold noise).")

    # Ensemble diversity: correlation of top-2 different-family OOFs.
    top_by_family = {}
    for fam, key in [("hgb", "_hgb_"), ("lgbm", "_lgbm_"), ("cb", "_cb_")]:
        fam_rows = [
            r for r in results
            if r["name"].startswith("v4") and key in r["name"] and "engineered" not in r["name"]
        ]
        # also catch v4a_hgb naming
        if fam == "hgb":
            fam_rows = [
                r for r in results
                if r["name"].startswith("v4") and "hgb" in r["name"] and "engineered" not in r["name"]
            ]
        elif fam == "lgbm":
            fam_rows = [
                r for r in results
                if r["name"].startswith("v4") and "lgbm" in r["name"] and "engineered" not in r["name"]
            ]
        elif fam == "cb":
            fam_rows = [
                r for r in results
                if r["name"].startswith("v4") and "_cb_" in r["name"] and "engineered" not in r["name"]
            ]
        if fam_rows:
            top_by_family[fam] = max(fam_rows, key=lambda r: r["oof_auc"])

    if len(top_by_family) >= 2:
        names = list(top_by_family.keys())
        corrs = []
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                a = oof_store[top_by_family[names[i]]["name"]]
                b = oof_store[top_by_family[names[j]]["name"]]
                corr = float(np.corrcoef(a, b)[0, 1])
                corrs.append((names[i], names[j], corr))
                print(f"OOF corr {names[i]} vs {names[j]}: {corr:.4f}")
        # Ensemble justified only if corr is not extremely high AND families are close.
        best_two = sorted(top_by_family.values(), key=lambda r: r["oof_auc"], reverse=True)[:2]
        gap = best_two[0]["oof_auc"] - best_two[1]["oof_auc"]
        min_corr = min(c for _, _, c in corrs)
        if min_corr < 0.98 and gap < 0.002 and not DATA_IS_SMOKE:
            promotion_decision["ensemble_justified"] = True
            print("Ensemble gate: OPEN (diverse + close families).")
        else:
            print(
                f"Ensemble gate: CLOSED (min_corr={min_corr:.4f}, gap={gap:.5f}, "
                f"smoke={DATA_IS_SMOKE})."
            )

    promotion_decision["gate_rows"] = gate_rows
    print("\nPromotion decision summary:")
    for k, v in promotion_decision.items():
        if k != "gate_rows":
            print(f"  {k}: {v}")
else:
    print("Promotion gate skipped.")

## 13. Phase 3 Summary / Next Moves


Record accepted and rejected experiments in
`docs/10_leaderboard_improvement_insights.md` and the narrative in
`docs/7_model_optimization_and_ensemble.md`.

Gates (from this run's evidence):
- **Optuna** — only if hand-search family spread exceeds fold noise.
- **XGBoost** — only if a later residual analysis shows complementary errors
  (not run by default).
- **Ensemble** — only if top families are close in OOF and not near-collinear.
- **Submission** — only from a non-smoke competition-data run via the public
  notebook (`docs/0_coding_standards.md`).


In [ ]:
# Final Phase 3 leaderboard table for the docs ledger.
if results:
    final = (
        pd.DataFrame(results)[["name", "oof_auc", "fold_auc_mean", "fold_auc_std"]]
        .sort_values("oof_auc", ascending=False)
        .reset_index(drop=True)
    )
    print(final.to_string(index=False))
    if DATA_IS_SMOKE:
        print("\n[SMOKE] Do not copy these AUCs into competition claim sections.")
else:
    print("No results collected.")